# 03 — Kilitli Tahminlerle Bağımsız Test Değerlendirmesi
Bu notebook **yalnızca** `test_predictions_locked.csv` GitHub'a ayrı bir commit ile yüklendikten sonra çalıştırılır. Test etiketleri burada ilk kez okunur. Bu aşamada **modele, eşiğe veya ön işleme kuralına müdahale edilmez**. En yüksek test skorunu veren seed ana sonuç olarak seçilmez; ana model önceden ilan edilmiştir.

In [ ]:
# === Kurulum: Google Colab (Drive'da ortak klasör) veya yerel Jupyter ===
import os, sys, subprocess
GITHUB_REPO_URL = "https://github.com/ilkinurcan/ysa-mlp-bank-marketing-ilkin.git"
DRIVE_DIR = "/content/drive/MyDrive/YSA_odev"   # üç notebook aynı klasörü kullanır

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs(DRIVE_DIR, exist_ok=True)
    repo_dir = os.path.join(DRIVE_DIR, GITHUB_REPO_URL.rstrip("/").split("/")[-1].removesuffix(".git"))
    if not os.path.exists(os.path.join(repo_dir, "src")):
        subprocess.run(["git", "clone", GITHUB_REPO_URL, repo_dir], check=True)
    else:  # repo'daki kod güncellemelerini al (üretilen sonuç dosyalarına dokunmaz)
        r = subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], capture_output=True, text=True)
        print("git pull:", (r.stdout or r.stderr).strip()[:300])
    os.chdir(repo_dir)
elif not os.path.exists("src"):
    raise RuntimeError("Repo klasöründe değilsin: os.chdir('<repo klasörü>') yap.")
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print("Çalışma klasörü:", os.getcwd())

In [ ]:
import hashlib, subprocess
import numpy as np, pandas as pd
from IPython.display import display
from src import config, data, plots
from src.config import RESULTS_DIR
from src.metrics import classification_metrics, bootstrap_ci, mean_sd, MAIN_METRICS

pd.set_option("display.width", 220); pd.set_option("display.max_columns", 40)
seeds = config.get_seeds()
final = config.load_json(RESULTS_DIR / "final_hyperparameters.json")
locked = pd.read_csv(RESULTS_DIR / "test_predictions_locked.csv")
sha_now = hashlib.sha256((RESULTS_DIR / "test_predictions_locked.csv").read_bytes()).hexdigest()
assert sha_now == final["locked_predictions_sha256"], "Kilitli tahmin dosyası değişmiş!"
assert "y" not in locked.columns
print("Kilitli dosya SHA-256 doğrulandı:", sha_now)
print("Ana model:", final["primary_model"], "| nihai aday:", final["final_config"]["id"],
      "| yöntem:", final["decision"]["final_method"], "| eşik:", final["threshold"])
try:
    subprocess.run(["git", "fetch", "origin"], capture_output=True)
    log = subprocess.run(["git", "log", "origin/HEAD", "--format=%h %ad %s", "--date=iso", "--",
                          "results/test_predictions_locked.csv"], capture_output=True, text=True).stdout.strip()
    print("Kilitli dosyanın git geçmişi:\n" + (log or "(git kaydı bulunamadı — commit'i yaptığından emin ol)"))
except Exception as e:
    print("git kontrolü yapılamadı:", e)

In [ ]:
# Test etiketleri İLK KEZ burada okunur
y_test_s, raw_test = data.load_test_labels()
y_true = y_test_s.loc[locked["original_index"]].to_numpy()
thr = final["threshold"]
primary = final["primary_model"]
models = {c.removesuffix("_proba"): c for c in locked.columns if c.endswith("_proba")}
rows = []
for name, col in models.items():
    m = classification_metrics(y_true, locked[col].to_numpy(), thr)
    rows.append({"model": name, "is_primary": name == primary, **m})
test_metrics = pd.DataFrame(rows)
test_metrics.to_csv(RESULTS_DIR / "test_metrics.csv", index=False)
print(f"Test n={len(y_true)}, pozitif={y_true.sum()} (%{100 * y_true.mean():.2f})")
test_metrics[["model", "is_primary"] + MAIN_METRICS + ["PR_AUC_trapz", "Brier", "TP", "FP", "TN", "FN"]].round(4)

In [ ]:
# Tablo 7: üç başlangıç seed'inin test performansı (betimsel) ve Mean ± SD
seed_df = test_metrics[test_metrics.model.str.startswith("mlp_seed")].copy()
seed_df["seed"] = seed_df.model.str.removeprefix("mlp_seed").astype(int)
seed_summary = mean_sd(seed_df.assign(grp="MLP 3 seed"), "grp")
seed_df.to_csv(RESULTS_DIR / "seed_stability_results.csv", index=False)
seed_summary.to_csv(RESULTS_DIR / "seed_stability_summary.csv", index=False)
display(seed_df[["seed", "is_primary"] + MAIN_METRICS].round(4))
display(seed_summary[["grp"] + MAIN_METRICS])
plots.seed_stability_plot(seed_df)

In [ ]:
# Tablo 6: bootstrap %95 yüzdelik güven aralıkları (1000 tekrar; tek sınıflı örneklemler yeniden çekilir)
ci_rows = []
for name in [primary, "logreg"]:
    ci, _ = bootstrap_ci(y_true, locked[models[name]].to_numpy(), n_boot=1000, seed=seeds["bootstrap"], threshold=thr)
    ci_rows.append(ci.assign(model=name))
ci = pd.concat(ci_rows, ignore_index=True)
ci.to_csv(RESULTS_DIR / "test_bootstrap_ci.csv", index=False)
ci.assign(GA=lambda d: [f"{p:.4f} [{a:.4f}, {b:.4f}]" for p, a, b in zip(d.point_estimate, d.ci_lower, d.ci_upper)])[
    ["model", "metric", "GA", "n_boot", "n_redrawn_single_class"]]

In [ ]:
pred = lambda n: (locked[models[n]].to_numpy() >= thr).astype(int)
plots.confusion_matrices({f"MLP ({primary})": (y_true, pred(primary)), "Logistic Regression": (y_true, pred("logreg"))})
tm = test_metrics.set_index("model")
labels = {f"MLP ({primary})": primary, "Logistic Regression": "logreg", "Dummy": "dummy"}
plots.roc_pr_curves(y_true, {k: locked[models[v]].to_numpy() for k, v in labels.items()},
                    ap_values={k: tm.loc[v, "AP"] for k, v in labels.items()},
                    auc_values={k: tm.loc[v, "ROC_AUC"] for k, v in labels.items()})

In [ ]:
# Başarısızlık analizi: FP / FN örnekleri (ana MLP)
p = locked[models[primary]].to_numpy(); yp = (p >= thr).astype(int)
kind = np.select([(y_true == 1) & (yp == 1), (y_true == 0) & (yp == 0), (y_true == 0) & (yp == 1), (y_true == 1) & (yp == 0)],
                 ["TP", "TN", "FP", "FN"], default="?")
err = raw_test.loc[locked["original_index"]].drop(columns=["duration", "y"]).assign(outcome=kind, proba=p)
counts = err.outcome.value_counts().reindex(["TP", "TN", "FP", "FN"]).to_frame("n")
display(counts)
num_cols = ["age", "campaign", "previous", "euribor3m", "nr.employed", "cons.conf.idx", "proba"]
num_err = err.groupby("outcome")[num_cols].median().reindex(["TP", "FN", "FP", "TN"]).round(3)
cat_err = pd.concat({c: pd.crosstab(err[c], err.outcome, normalize="columns").round(3)
                     for c in ["poutcome", "contact", "month"]})
num_err.to_csv(RESULTS_DIR / "error_analysis_numeric_medians.csv")
cat_err.to_csv(RESULTS_DIR / "error_analysis_categorical.csv")
counts.to_csv(RESULTS_DIR / "error_analysis_counts.csv")
print("Sayısal medyanlar (sonuç türüne göre):"); display(num_err)
print("Kategorik dağılımlar (sütun içi oran):"); display(cat_err)
print("FN olasılık dağılımı:"); display(err.loc[err.outcome == "FN", "proba"].describe().round(3))

In [ ]:
# İsteğe bağlı (betimsel): kalibrasyon ve Brier skoru — test ile hiçbir şey optimize edilmez
plots.calibration_plot(y_true, {f"MLP ({primary})": locked[models[primary]].to_numpy(), "Logistic Regression": locked[models["logreg"]].to_numpy()})
test_metrics[["model", "Brier"]]

In [ ]:
# Üç seviyenin ayrı yorumlanması: outer CV (prosedür), development inner CV (seçim), bağımsız test
outer = pd.read_csv(RESULTS_DIR / "outer_cv_results.csv")
method_map = {"manual_search": "MLP_manual_search", "random_search": "MLP_random_search"}
o = outer[outer.method == method_map[final["decision"]["final_method"]]]
levels = pd.DataFrame([
    {"seviye": "Outer CV (seçilen arama prosedürü, n=3)", "AP": f"{o.AP.mean():.4f} ± {o.AP.std(ddof=1):.4f}",
     "ROC_AUC": f"{o.ROC_AUC.mean():.4f} ± {o.ROC_AUC.std(ddof=1):.4f}"},
    {"seviye": f"Development inner CV ({final['final_config']['id']})",
     "AP": f"{final['decision'][final['decision']['final_method'].split('_')[0] + '_best_inner_AP_mean']:.4f}", "ROC_AUC": "-"},
    {"seviye": f"Bağımsız test ({primary})", "AP": f"{tm.loc[primary, 'AP']:.4f}", "ROC_AUC": f"{tm.loc[primary, 'ROC_AUC']:.4f}"},
    {"seviye": "Bağımsız test (Logistic Regression)", "AP": f"{tm.loc['logreg', 'AP']:.4f}", "ROC_AUC": f"{tm.loc['logreg', 'ROC_AUC']:.4f}"},
])
levels.to_csv(RESULTS_DIR / "three_level_summary.csv", index=False)
levels

In [ ]:
import shutil
for d in ["results", "figures"]:
    shutil.make_archive(f"export_final_{d}", "zip", root_dir=".", base_dir=d)
if IN_COLAB:
    from google.colab import files
    for d in ["results", "figures"]:
        files.download(f"export_final_{d}.zip")
print("Bitti. results/ ve figures/ klasörlerini GitHub'a gönder; raporu bu dosyalardaki sayılarla yaz.")